# Machine Learning with Spark

We will predict the **price** of a flat based on:
- `bedrooms`
- `surface`
- `floors`

## 1. Download the data

In [ ]:
!curl https://wagon-public-datasets.s3.amazonaws.com/flats.csv > flats.csv

## 2. Create a Spark session

In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MLExample").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 00:12:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 3. Load and explore the data

In [3]:
data = spark.read.csv("flats.csv", header=True, inferSchema=True)
data.show()

+------+--------+-------+------+
| price|bedrooms|surface|floors|
+------+--------+-------+------+
| 274.0|       3|   1830|   2.0|
| 500.0|       4|   2120|   1.0|
| 320.0|       3|   1260|   1.0|
| 445.5|       3|   1880|   1.0|
| 637.5|       3|   1680|   1.0|
| 460.0|       2|   2730|   1.0|
| 259.0|       3|   1270|   1.5|
| 950.0|       3|   2780|   1.0|
| 550.0|       3|   1930|   2.0|
| 265.5|       3|   1860|   1.0|
| 162.0|       4|   1460|   1.0|
|2395.0|       4|   3800|   2.0|
| 385.0|       3|   1070|   1.0|
| 230.0|       3|   1010|   1.0|
| 665.0|       3|   1940|   1.5|
| 412.0|       4|   3360|   2.0|
| 177.5|       3|   1220|   1.0|
| 330.0|       4|   2000|   1.0|
| 445.0|       4|   2430|   1.5|
| 139.5|       2|   1230|   2.0|
+------+--------+-------+------+
only showing top 20 rows



In [4]:
data.printSchema()

root
 |-- price: double (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- surface: integer (nullable = true)
 |-- floors: double (nullable = true)



## 4. Convert the features

Spark ML needs all features combined into a single vector column. That's what `VectorAssembler` does.

In [5]:
from pyspark.ml.feature import VectorAssembler

vector_assembler = VectorAssembler(
    inputCols=["bedrooms", "surface", "floors"],
    outputCol="features"
)

data_assembled = vector_assembler.transform(data)
data_assembled.show()

+------+--------+-------+------+----------------+
| price|bedrooms|surface|floors|        features|
+------+--------+-------+------+----------------+
| 274.0|       3|   1830|   2.0|[3.0,1830.0,2.0]|
| 500.0|       4|   2120|   1.0|[4.0,2120.0,1.0]|
| 320.0|       3|   1260|   1.0|[3.0,1260.0,1.0]|
| 445.5|       3|   1880|   1.0|[3.0,1880.0,1.0]|
| 637.5|       3|   1680|   1.0|[3.0,1680.0,1.0]|
| 460.0|       2|   2730|   1.0|[2.0,2730.0,1.0]|
| 259.0|       3|   1270|   1.5|[3.0,1270.0,1.5]|
| 950.0|       3|   2780|   1.0|[3.0,2780.0,1.0]|
| 550.0|       3|   1930|   2.0|[3.0,1930.0,2.0]|
| 265.5|       3|   1860|   1.0|[3.0,1860.0,1.0]|
| 162.0|       4|   1460|   1.0|[4.0,1460.0,1.0]|
|2395.0|       4|   3800|   2.0|[4.0,3800.0,2.0]|
| 385.0|       3|   1070|   1.0|[3.0,1070.0,1.0]|
| 230.0|       3|   1010|   1.0|[3.0,1010.0,1.0]|
| 665.0|       3|   1940|   1.5|[3.0,1940.0,1.5]|
| 412.0|       4|   3360|   2.0|[4.0,3360.0,2.0]|
| 177.5|       3|   1220|   1.0|[3.0,1220.0,1.0]|


## 5. Train the model

Linear Regression: predict `price` based on the `features` vector.

In [6]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="price")
lr_model = lr.fit(data_assembled)

26/03/20 00:13:39 WARN Instrumentation: [e02d8c49] regParam is zero, which might cause numerical instability and overfitting.
26/03/20 00:13:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/20 00:13:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


## 6. Evaluate the model

In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = lr_model.transform(data_assembled)
evaluator = RegressionEvaluator(labelCol="price")

rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
print(f"Root Mean Squared Error (RMSE): {rmse}")

mae = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
print(f"Mean Absolute Error (MAE): {mae}")

r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})
print(f"R^2: {r2}")

Root Mean Squared Error (RMSE): 240.50644355361382
Mean Absolute Error (MAE): 164.4301643578809
R^2: 0.5422973289292989


## 7. Check the predictions

In [8]:
predictions.select("bedrooms", "surface", "floors", "price", "prediction").show()

+--------+-------+------+------+------------------+
|bedrooms|surface|floors| price|        prediction|
+--------+-------+------+------+------------------+
|       3|   1830|   2.0| 274.0|470.78499014020696|
|       4|   2120|   1.0| 500.0|  536.189754989064|
|       3|   1260|   1.0| 320.0|311.03346895343486|
|       3|   1880|   1.0| 445.5|488.94452380677603|
|       3|   1680|   1.0| 637.5|431.55386095085953|
|       2|   2730|   1.0| 460.0| 754.4784051892331|
|       3|   1270|   1.5| 259.0|311.99706811993576|
|       3|   2780|   1.0| 950.0| 747.2025066584004|
|       3|   1930|   2.0| 550.0|499.48032156816527|
|       3|   1860|   1.0| 265.5|483.20545752118437|
|       4|   1460|   1.0| 162.0| 346.8005675645395|
|       4|   3800|   2.0|2395.0|1014.4594550261727|
|       3|   1070|   1.0| 385.0| 256.5123392403142|
|       3|   1010|   1.0| 230.0|239.29514038353923|
|       3|   1940|   1.5| 665.0| 504.2557886872561|
|       4|   3360|   2.0| 412.0| 888.1999967431564|
|       3|  

## 8. Stop the session

In [ ]:
spark.stop()